In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# XGBClassifier
>欠損値（NaN）の処理 → 内部で最適な分岐先を学習してくれる

>スケーリング不要 → 決定木ベースなので、値の大きさは関係ない

>数値列のビニング等 → 必要ない
>
こちらがやることとして、特徴量エンジニアリングと、カテゴリ列のエンコードをやっていく。追加する特徴量などは、04_baselineと同じ。

# データ読み込み

In [2]:
test_csv = pd.read_csv("/kaggle/input/competitions/titanic/test.csv").set_index("PassengerId")
train_csv = pd.read_csv("/kaggle/input/competitions/titanic/train.csv").set_index("PassengerId")

# trainデータへの特徴量の追加

In [3]:
train_csv["Sex_Pclass"] = train_csv["Sex"] + "_" + train_csv["Pclass"].astype(str)

train_csv["logFare"] = np.log1p(train_csv["Fare"])

train_csv["Family"] = train_csv[["Parch","SibSp"]].sum(axis=1)
train_csv["len_Fam"] = train_csv["Family"].apply(lambda x:"alone" if x==0 else "basic" if 1<=x<=3 else "large")
train_csv=train_csv.drop("Family",axis=1)

# テストデータへの特徴量の追加

In [4]:
test_csv["Sex_Pclass"] = test_csv["Sex"] + "_" + test_csv["Pclass"].astype(str)

test_csv["logFare"] = np.log1p(test_csv["Fare"])

test_csv["Family"] = test_csv[["Parch","SibSp"]].sum(axis=1)
test_csv["len_Fam"] = test_csv["Family"].apply(lambda x:"alone" if x==0 else "basic" if 1<=x<=3 else "large")
test_csv=test_csv.drop("Family",axis=1)

# 特徴量の選定

In [5]:
y = train_csv.Survived
X = train_csv.drop("Survived",axis=1)

cat_cols = [c for c in X.columns if X[c].dtype in ["object","category"] and X[c].nunique() <=10]
X[cat_cols] = X[cat_cols].astype("category")

num_cols = [c for c in X.columns if X[c].dtype in ["int64","float64"]]

using_cols = cat_cols + num_cols
X = X[using_cols].copy()

# データ分割

In [6]:
from sklearn.model_selection import train_test_split

train_X,valid_X,train_y,valid_y = train_test_split(
    X,y,test_size=0.2,random_state=0
)

# モデル作成
とりあえずのパラメータ値を設定。
early_stopping_roundsは、昔の環境ではfitの方に書かれていることがあるが、今はこっちのconstructor側に書かないとおかしくなる。

また、後述するcross_val_scoreでは、early_stopping_roundsを書くと「Early Stoppingを使うなら、検証データ(eval_set)を渡してください。」といったエラーが起こる。

つまり、eval_setが無いearly_stoppingは無効になる。

In [7]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators = 1000,
    learning_rate = 0.05,
    max_depth = 5,
    gamma = 1,
    min_child_weight = 2,
    subsample = 0.8,
    colsample_bytree = 0.8,
    enable_categorical = True,
    #early_stopping_rounds = 10,
    #cross_val_scoreのためにいったん消す
    random_state = 10
)



# cross_val_scoreで検証

In [8]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model,X,y,cv=5,scoring="accuracy")
print(scores.mean())

0.832803967108154


正解率83%なので、多かい数字が出ていることがわかる。

In [9]:
model = XGBClassifier(
    n_estimators = 1000,
    learning_rate = 0.05,
    max_depth = 5,
    gamma = 1,
    min_child_weight = 2,
    subsample = 0.8,
    colsample_bytree = 0.8,
    enable_categorical = True,
    early_stopping_rounds = 10,
    random_state = 10
)

model.fit(
    train_X,train_y,
    eval_set=[(valid_X,valid_y)],
    #early_stopping_rounds = 10,こちらには書かない。環境によってはこっちに書く
    verbose = False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=10,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=1, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=2, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_parallel_tree=None, ...)

# 提出物の作成

In [10]:
#testデータのobject列をカテゴリ列に変換しておく
cat_cols = [c for c in test_csv.columns if test_csv[c].dtype in ["object","category"] and test_csv[c].nunique() <=10]
test_csv[cat_cols] = test_csv[cat_cols].astype("category")
num_cols = [c for c in test_csv.columns if test_csv[c].dtype in ["int64","float64"]]
using_cols = cat_cols + num_cols


preds = model.predict(test_csv[using_cols])
output = pd.DataFrame({"PassengerId":test_csv.index,"Survived":preds})
output.to_csv("submission.csv",index=False)